# Data Quality Check(DQ) -- Assertions --

In [0]:
# 1. Row count > 0 — the table actually loaded
assert spark.table("silver.orders").count() > 0, "silver.orders is empty"



In [0]:
# 2. PK uniqueness — no duplicate primary keys survived Silver dedup
total = spark.table("silver.orders").count()
distinct = spark.table("silver.orders").select("order_id").distinct().count()
assert total == distinct, f"Duplicate order_id in silver.orders: {total} rows, {distinct} unique"

In [0]:
total = spark.table("silver.order_items").count()
distinct = spark.table("silver.order_items").select("order_id", "order_item_id").distinct().count()
assert total == distinct, f"Duplicate (order_id, order_item_id) in silver.order_items: {total} vs {distinct}"

In [0]:
orphans = (
    spark.table("silver.order_items")
    .join(spark.table("silver.orders"), "order_id", "left_anti")
    .count()
)
assert orphans == 0, f"{orphans} order_items reference a non-existent order"

In [0]:
assert spark.table("silver.order_items").count() > 0, "silver.order_items is empty"

assert spark.table("silver.orders").filter("order_id IS NULL").count() == 0, "null order_id in silver.orders"
assert spark.table("silver.order_items").filter("order_id IS NULL").count() == 0, "null order_id in silver.order_items"